<a href="https://colab.research.google.com/github/Layaa-V/MLOps-LayaaVishwakarma-M25CSA017/blob/Lab-2-Worksheet/LayaaVishwakarma_M25CSA017_lab2_Worksheet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install wandb thop

  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 175.8 kB/s  0:02:04
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 190.6 kB/s  0:00:10
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12 [wandb]
Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import wandb
import time
from thop import profile, clever_format

In [7]:
class CIFAR10Dataset(torch.utils.data.Dataset):
    def __init__(self, root, train=True, transform=None):
        self.dataset = torchvision.datasets.CIFAR10(
            root=root, train=train, download=True, transform=None
        )
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

def get_dataloaders(batch_size=128):
    # CIFAR-10 stats
    stats = ((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))

    # Transformations
    train_transform = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(*stats)
    ])

    test_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(*stats)
    ])

    train_ds = CIFAR10Dataset(root='./data', train=True, transform=train_transform)
    test_ds = CIFAR10Dataset(root='./data', train=False, transform=test_transform)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2)

    return train_loader, test_loader

In [8]:
def get_resnet_cifar(device):
    model = models.resnet18(weights=None)

    # Modify for 32x32 image size (Prevent feature map vanishing)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(512, 10)

    return model.to(device)

In [10]:
def get_grad_flow_fig(named_parameters):
    ave_grads = []
    max_grads = []
    layers = []

    for n, p in named_parameters:
        if(p.requires_grad) and ("bias" not in n) and p.grad is not None:
            # Shorten names for cleaner plots
            if "conv" in n or "fc" in n or "downsample" in n:
                name_short = n.replace("layer","L").replace("weight","").replace("module.","")
                layers.append(name_short)
                ave_grads.append(p.grad.abs().mean().cpu().item())
                max_grads.append(p.grad.abs().max().cpu().item())

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(np.arange(len(max_grads)), max_grads, alpha=0.3, lw=1, color="c", label="Max Grad")
    ax.bar(np.arange(len(max_grads)), ave_grads, alpha=0.8, lw=1, color="b", label="Avg Grad")
    ax.hlines(0, 0, len(ave_grads), lw=2, color="k")
    ax.set_xticks(np.arange(len(ave_grads)))
    ax.set_xticklabels(layers, rotation="vertical", fontsize=8)
    ax.set_xlabel("Layers")
    ax.set_ylabel("Gradient Magnitude")
    ax.set_title("Gradient Flow")
    ax.legend()
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    return fig

In [11]:
def get_weight_update_fig(model, old_weights, lr):
    updates = []
    layers = []

    for name, param in model.named_parameters():
        if param.requires_grad and "weight" in name and "bn" not in name:
            if name in old_weights:
                # Calculate L2 norm of the update step
                new_w = param.data
                old_w = old_weights[name]
                diff = (new_w - old_w).norm().item()
                updates.append(diff)
                layers.append(name.replace("layer","L").replace(".weight",""))

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(np.arange(len(updates)), updates, color="orange", alpha=0.7)
    ax.set_xticks(np.arange(len(updates)))
    ax.set_xticklabels(layers, rotation="vertical", fontsize=8)
    ax.set_xlabel("Layers")
    ax.set_ylabel("Update Magnitude (L2 Norm)")
    ax.set_title(f"Weight Update Flow (Approx per epoch)")
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    return fig

In [12]:
config = {
        "epochs": 25,
        "batch_size": 128,
        "lr": 0.01,
        "model": "ResNet18-CIFAR",
        "dataset": "CIFAR-10"}

In [13]:
wandb.init(project="cifar10-assignment-lab2", config=config)

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /DATA/anikde/.netrc
wandb: Currently logged in as: vishwakarmalayaa (vishwakarmalayaa-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [22]:
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
print(f"Running on device: {device}")

Running on device: cuda:1


In [24]:
train_loader, test_loader = get_dataloaders(config["batch_size"])
model = get_resnet_cifar(device)

Files already downloaded and verified
Files already downloaded and verified


In [25]:
dummy_input = torch.randn(1, 3, 32, 32).to(device)
flops, params = profile(model, inputs=(dummy_input,), verbose=False)
flops_str, params_str = clever_format([flops, params], "%.3f")

print(f"FLOPs: {flops_str}, Params: {params_str}")
# Log FLOPs to WandB Summary
wandb.run.summary["total_flops"] = flops
wandb.run.summary["total_params"] = params
wandb.run.summary["flops_readable"] = flops_str

FLOPs: 557.889M, Params: 11.174M


In [26]:
wandb.watch(model, log="all", log_freq=50)

# Setup Training
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=config["lr"], momentum=0.9, weight_decay=5e-4)

# Store initial weights for update comparison
old_weights = {name: p.data.clone() for name, p in model.named_parameters()}

In [27]:
for epoch in range(config["epochs"]):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (inputs, targets) in enumerate(train_loader):
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()

        optimizer.step()

        # Track Accuracy
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
        running_loss += loss.item()

        # Log Batch Metrics to WandB
        wandb.log({
            "batch_loss": loss.item(),
            "batch_acc": 100. * correct / total,
            "epoch": epoch
        })

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100. * correct / total

    grad_fig = get_grad_flow_fig(model.named_parameters())
    update_fig = get_weight_update_fig(model, old_weights, config["lr"])

    # Log everything to WandB
    wandb.log({
        "train_loss": epoch_loss,
        "train_accuracy": epoch_acc,
        "gradient_flow_chart": wandb.Image(grad_fig),
        "weight_update_chart": wandb.Image(update_fig)
    })

    # Cleanup plots to save memory
    plt.close(grad_fig)
    plt.close(update_fig)

    # Update "old_weights" for the next epoch comparison
    old_weights = {name: p.data.clone() for name, p in model.named_parameters()}

wandb.finish()

batch_acc,▂▁▃▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇█████████████████
batch_loss,█▇▇▅▆▆▄▄▃▃▂▂▂▂▂▃▃▂▂▂▂▂▂▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▂
epoch,▁▁▁▁▁▂▂▂▂▂▂▂▂▂▂▂▂▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇███
train_accuracy,▁▁▃▅▅▆▆▆▇▇▇▇▇▇▇▇██████████
train_loss,██▆▅▄▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
batch_acc,93.824
batch_loss,0.27556
epoch,24
flops_readable,557.889M
total_flops,557889024.0
total_params,11173962.0
